# Google ADK(Agent Development Kit)
Google ADK (Agent Development Kit)는 복잡한 AI 에이전트와 멀티 에이전트 시스템(MAS)을 더 쉽고 체계적으로 개발하기 위해 Google에서 공개한 오픈소스 프레임워크입니다.
기존의 단순한 챗봇을 넘어, 스스로 판단하고 도구를 사용하며 협업하는 '자율형 에이전트'를 구축하는 데 최적화되어 있습니다.

🚀 주요 특징
- 프레임워크 독립성 (Model-Agnostic): Gemini와 Google 생태계에 최적화되어 있지만, 다른 AI 모델이나 프레임워크와도 호환되도록 설계되었습니다.
- 멀티 에이전트 시스템(MAS) 지원: 여러 개의 에이전트가 계층 구조를 이루어 협업하거나 각자의 전문 분야를 나누어 복잡한 목표를 달성할 수 있게 합니다.
- 에이전트 간 통신 (Agent2Agent, A2A): 서로 다른 프레임워크나 환경에서 구축된 에이전트들이 표준화된 프로토콜로 소통할 수 있도록 지원합니다.
- 상태 및 메모리 관리: 대화의 맥락을 유지하기 위한 세션 관리 및 Firestore, Vertex AI 기반의 메모리 서비스를 제공합니다.

🛠️ 핵심 구성 요소 및 도구
ADK는 개발자가 직접 바닥부터 코딩하는 수고를 덜어주는 다양한 내장 기능을 포함합니다.
- Skills (기술): 에이전트가 수행할 수 있는 특정 기능 단위입니다. '인라인 기술'로 간단히 정의하거나, 재사용 가능한 파일 형태로 관리할 수 있습니다.
- Built-in Tools 연동: 앞서 설명드린 Google Search, Google Maps, Code Execution 등이 ADK 내에 도구(Tool) 형태로 통합되어 있어 즉시 연결이 가능합니다.
- Human-in-the-Loop (HITL): 중요한 결정이나 도구 실행 전 사람의 승인을 기다리는 워크플로우를 지원합니다.
- Sandbox 환경: 안전한 코드 실행을 위해 격리된 환경에서 에이전트가 작업을 수행하도록 돕습니다.

## 기본 Agent 구현

In [1]:
# 패키지 설치
!pip install google-adk

###  ADK Agent 예제 사용법 
1. TestAgent 폴더를 만들고 아래 두 셀을 차례로 실행시켜 생성된 2개의 파이썬 소스를 이 폴더안에 복사해놓는다
2. 최상위 폴더에 .env 파일을 만들고 아래 내용을 설정해 넣는다(프로젝트에 자신의 프로젝트 ID를 입력하여 사용한다)
   
GOOGLE_GENAI_USE_VERTEXAI=TRUE

GOOGLE_CLOUD_PROJECT=byounghwa-go

GOOGLE_CLOUD_LOCATION=global

4. Launcher 탭에서 Terminal을 실행 시킨 다음 아래 명령을 실행한다
   
adk run TestAgent

5.아래와 같이 [[user]] 에 입력하여 결과를 확인해본다.  exit를 입력하면 종료된다

Running agent weather_time_agent, type exit to exit.

[[user]]: 현재 서울의 날씨 알려줘

[weather_time_agent]: 서울의 날씨는 맑고 기온은 25도 입니다.섭씨 기준.(화씨 77도)

[[user]]: 현재 서울의 시간 알려줘

[weather_time_agent]: 서울 도시의 현재 시간은 2026-04-13 20:59:48 KST+0900 입니다.

[user]: exit

In [4]:
%%writefile __init__.py
from . import agent

Writing __init__.py


In [2]:
%%writefile agent.py
import datetime
from zoneinfo import ZoneInfo
from google.adk.agents import Agent

# 도구1. 특정 도시의 날씨 정보를 가져옵니다.
def get_weather(city: str) -> dict:
    """특정 도시의 현재 날씨 정보를 검색합니다.
    Args:
        city (str): 날씨 정보를 검색할 도시 이름
    Returns:
        dict: 검색 결과
    """
    if city.lower() == "서울":
        return {
            "status": "success",
            "report": (
                "서울의 날씨는 맑고 기온은 25도 입니다."
                "섭씨 기준.(화씨 77도)"
            ),
        }
    else:
        return {
            "status": "error",
            "error_message": f"'{city}' 도시의 날씨는 검색할 수 없습니다."
        }

# 도구2. 특정 도시의 현재 시간을 가져옵니다.
def get_current_time(city: str) -> dict:
    """Returns the current time in a specified city.
    Args:
        city (str): 현재 시간을 검색할 도시 이름
    Returns:
        dict: 검색 결과
    """
    if city.lower() == "서울":
        tz_identifier = "Asia/Seoul"
    else:
        return {
            "status": "error",
            "error_message": (
                f"{city} 도시의 현재 시간을 가져올 수 없습니다."
            ),
        }
    tz = ZoneInfo(tz_identifier)
    now = datetime.datetime.now(tz)
    report = (
        f'{city} 도시의 현재 시간은 {now.strftime("%Y-%m-%d %H:%M:%S %Z%z")} 입니다.'
    )
    return {"status": "success", "report": report}

# LlmAgent 정의
root_agent = Agent(
    name="weather_time_agent",
    model="gemini-3-flash-preview", # 사용할 Gemini 모델 지정
    description="도시의 실시간 날씨 및 시간을 조회할 수 있는 에이전트입니다.",
    instruction=(
        "당신은 사용자에의 질문에서 도시를 식별하고, 해당 도시의 실시간 날씨 및 시간을 조회할 수 있는 에이전트입니다."
        "사용자가 날씨를 물어보는 경우, 'get_weather' 도구를 호출하여 검색하고, 현재 시간을 물어보는 경우 'get_current_time' 도구를 호출하여 조회하세요."
    ),
    tools=[get_weather, get_current_time], # 에이전트가 사용할 도구 목록)
)    

Writing agent.py


### Multi-Agent(멀티 에이전트)
Multi-Agent(멀티 에이전트)란 말 그대로 "여러 명의 지능형 에이전트가 팀을 이뤄 서로 협업하며 복잡한 문제를 해결하는 시스템"을 의미합니다.

혼자서 모든 일을 처리하는 '만능 천재(Single Agent)'를 만드는 대신, 특정 분야에 특화된 '전문가 팀'을 구성한다고 생각하면 이해가 빠릅니다.

1. 멀티 에이전트의 핵심 구성 요소
멀티 에이전트 시스템(MAS, Multi-Agent System)은 보통 다음과 같은 방식으로 움직입니다.
- Role (역할 분담): 각 에이전트에게 명확한 페르소나와 임무를 줍니다. (예: "너는 검색 전문가야", "너는 코드 리뷰어고", "너는 최종 보고서 작성자야")
- Collaboration (협업): 에이전트끼리 대화(A2A 통신)를 주고받으며 결과를 넘겨줍니다.
- Orchestration (조율): 보통 '매니저' 역할을 하는 에이전트가 전체 흐름을 관리하고 일을 배분합니다.

2. 왜 멀티 에이전트인가? (장점)
단일 에이전트(단일 LLM)로 처리할 때보다 다음과 같은 강점이 있습니다.

- 정확도 향상: 한 명의 에이전트가 너무 많은 지시사항(Prompt)을 받으면 헷갈려 하지만, 역할을 쪼개면 각 작업의 퀄리티가 올라갑니다.
- 검증 가능 (Self-Correction): '작성 에이전트'가 쓴 글을 '검토 에이전트'가 비판하고 수정하는 루프를 만들 수 있습니다. (환각 현상 감소)
- 복잡한 워크플로우 처리: 코딩, 테스트, 배포처럼 여러 단계가 필요한 업무를 자동화하기에 최적입니다.

###  Multi Agent 예제 사용법 
1. MultiAgent 폴더를 만들고 아래 두 셀을 차례로 실행시켜 생성된 2개의 파이썬 소스를 이 폴더안에 복사해놓는다
2. 최상위 폴더에 .env 파일을 만들고 아래 내용을 설정해 넣는다(프로젝트에 자신의 프로젝트 ID를 입력하여 사용한다)
   
GOOGLE_GENAI_USE_VERTEXAI=TRUE

GOOGLE_CLOUD_PROJECT=byounghwa-go

GOOGLE_CLOUD_LOCATION=global

4. Launcher 탭에서 Terminal을 실행 시킨 다음 아래 명령을 실행한다
   
adk run MultiAgent

5.아래와 같이 [[user]] 에 입력하여 결과를 확인해본다.  exit를 입력하면 종료된다

Running agent ResearchSynthesisPipeline, type exit to exit.

[[user]]: 최신 전기차 기술에 대하여 알려줘

[EVResearcher]: 최신 전기차 기술은 배터리 성능, 충전 인프라, 모터 효율성 및 스마트 기능 발전에 중점을 두고 있습니다.  ....

[RenewableEnergyResearcher]: 최신 전기차 기술은 배터리, 충전, 모터, 자율주행 등 다방면에서 빠르게 발전하고 있습니다. ....

[CarbonCaptureResearcher]: 최신 전기차 기술은 주로 배터리 성능 향상, 충전 기술 혁신, 그리고 모터 효율성 증대에 중점을 두고 빠르게 발전하고 있습니다

[SynthesisAgent]: ## 최신 지속 가능한 기술 발전 요약

신재생 에너지 연구에 따르면, 최신 전기차 기술은 배터리 .....

[user]]:  exit


In [6]:
%%writefile agent.py
from google.adk.agents import Agent, LlmAgent, SequentialAgent, ParallelAgent
from google.adk.tools import google_search # Google Search 도구 임포트

# 1. 연구원 에이전트 A: 신재생 에너지
researcher_agent_1 = LlmAgent(
    name="RenewableEnergyResearcher",
    model="gemini-2.5-flash",
    instruction="""당신은 에너지 전문 AI 연구 보조원입니다.
    '신재생 에너지원'의 최신 발전에 대해 Google Search 도구를 사용하여 조사하세요.
    핵심 조사 결과를 간결하게 요약하세요 (1-2문장).
    """,
    description="신재생 에너지원을 조사합니다.",
    tools=[google_search],
    output_key="renewable_energy_result"
)

# 2. 연구원 에이전트 B: 전기차 기술
researcher_agent_2 = LlmAgent(
    name="EVResearcher",
    model="gemini-2.5-flash",
    instruction="""당신은 운송 기술 전문 AI 연구 보조원입니다.
    '전기차 기술'의 최신 개발 동향에 대해 Google Search 도구를 사용하여 조사하세요.
    핵심 조사 결과를 간결하게 요약하세요 (1-2문장).
    """,
    description="전기차 기술을 조사합니다.",
    tools=[google_search],
    output_key="ev_technology_result"
)

# 연구원 에이전트 C: 탄소 포집 방법
researcher_agent_3 = LlmAgent(
    name="CarbonCaptureResearcher",
    model="gemini-2.5-flash",
    instruction="""당신은 기후 솔루션 전문 AI 연구 보조원입니다.
    '탄소 포집 방법'의 현재 상태에 대해 Google Search 도구를 사용하여 조사하세요.
    핵심 조사 결과를 간결하게 요약하세요 (1-2문장).
    """,
    description="탄소 포집 방법을 조사합니다.",
    tools=[google_search],
    output_key="carbon_capture_result"
)

# 병렬 에이전트(ParallelAgent): 동시 연구 진행
parallel_research_agent = ParallelAgent(
    name="ParallelWebResearchAgent",
    sub_agents=[researcher_agent_1, researcher_agent_2, researcher_agent_3],
    description="여러 연구 에이전트를 병렬로 실행하여 정보를 수집합니다."
)

# 병합 에이전트: 병렬 에이전트들이 세션 상태에 저장한 결과들을 취합하여 하나의 구조화된 보고서로 종합.
merger_agent = LlmAgent(
    name="SynthesisAgent",
    model="gemini-2.5-flash",
    instruction="""당신은 연구 결과를 구조화된 보고서로 결합하는 AI 도우미입니다.
    당신의 주요 임무는 다음 연구 요약들을 종합하여, 각 발견의 출처 영역을 명확히 밝히는 것입니다.
    각 주제별로 제목을 사용하여 응답을 구조화하세요. 보고서가 일관성 있고 핵심 내용을 매끄럽게 통합하도록 하세요.

    결정적으로 당신의 전체 응답은 오직 아래 'SUMMARY'에 제공된 정보에만 기반해야 합니다.
    여기에 없는 외부 지식, 사실, 또는 세부 사항을 추가하지 마세요.
    반드시 아래의 OUTPUT_FORMAT에 따라 보고서를 생성하세요.

    SUMMARY:
    *   **신재생 에너지:** {renewable_energy_result}
    *   **전기차:** {ev_technology_result}
    *   **탄소 포집:** {carbon_capture_result}

    OUTPUT_FORMAT:
    ## 최신 지속 가능한 기술 발전 요약
    ### 신재생 에너지 연구 결과
    (위에서 제공된 신재생 에너지 SUMMARY에 대해서 종합하여 설명하세요.)

    ### 전기차 연구 결과
    (위에서 제공된 전기차 SUMMARY에 대해서 종합하여 설명하세요.)

    ### 탄소 포집 연구 결과
    (위에서 제공된 탄소 포집 SUMMARY에 대해서 종합하여 설명하세요.)

    ### 전반적인 결론
    (위에서 제공된 전체 연구 결과에 대해서 짧게 1~2줄로 종합하여 요약하세요.)
    """,
    description="병렬 에이전트의 연구 결과를 구조화된 보고서로 결합하고, 제공된 입력에 엄격히 근거합니다.",
)

# 순차 에이전트(SequentialAgent): 전체 순차 파이프라인
root_agent = SequentialAgent(
    name="ResearchSynthesisPipeline",
    sub_agents=[parallel_research_agent, merger_agent], # 병렬 연구 먼저 실행, 그 다음 병합
    description="병렬 연구를 조율하고 결과를 종합합니다."
)

Overwriting agent.py


## Agent 배포(Deploy)  --> 배포 소스 오류나므로 Agent 실행까지만 실습한다!!
Google ADK(Agent Development Kit)로 개발한 에이전트를 실제 운영 환경으로 내보내는 3가지 주요 배포 옵션을 정리해 드립니다.
ADK는 기본적으로 '에이전트 로직'을 코드로 정의하기 때문에, 이를 어디에 올리느냐에 따라 서비스의 확장성과 관리 편의성이 달라집니다.

### 1. Cloud Run 배포 (API 중심형)
가장 보편적이고 권장되는 방식입니다. 에이전트를 컨테이너화하여 서버리스로 운영합니다.
- 방식: 에이전트 코드를 Docker 이미지로 빌드하여 Cloud Run에 배포하고, 외부에서 호출할 수 있는 **HTTPS 엔드포인트(API)**를 생성합니다.
- 특징:
비용 효율성: 요청이 있을 때만 서버가 가동되므로 비용이 저렴합니다.
유연성: 특정 웹사이트, 모바일 앱, 혹은 Slack/Teams 같은 메신저에 API 형태로 쉽게 연동할 수 있습니다.
-추천 상황: 에이전트를 독립적인 백엔드 서비스로 구축하고 싶을 때.

### 2. Vertex AI Agent Builder 연동 (플랫폼 통합형)
구글 클라우드의 관리형 에이전트 서비스인 Agent Builder와 결합하는 방식입니다.
- 방식: ADK로 만든 에이전트를 Agent Builder의 'Custom Engine' 또는 **'Remote Tool'**로 등록합니다.
- 특징:
GUI 제공: 구글이 제공하는 기본 채팅 위젯(Web Widget)을 그대로 사용할 수 있어 프론트엔드 개발 부담이 적습니다.
데이터 연동: Vertex AI Search와 연동하여 기업 내부 데이터를 근거(Grounding)로 사용하는 설정이 매우 간편합니다.
분석 도구: 대화 내역 저장, 만족도 조사, 성능 모니터링 대시보드를 기본으로 제공합니다.
- 추천 상황: 빠른 서비스 출시가 필요하거나, 구글 클라우드의 관리 기능을 최대한 활용하고 싶을 때.

### 3. GKE (Google Kubernetes Engine) 배포 (엔터프라이즈 확장형)
대규모 멀티 에이전트 시스템이나 복잡한 인프라 제어가 필요한 경우 사용합니다.
- 방식: 쿠버네티스 클러스터 위에서 여러 에이전트 컨테이너를 관리하며, 에이전트 간의 통신(A2A)을 정교하게 제어합니다.
- 특징:
고가용성: 대규모 트래픽 처리에 최적화되어 있으며, GPU/TPU 자원을 직접 할당할 수 있습니다.
복합 구성: 여러 명의 전문 에이전트(작성, 검토, 검색 등)가 복합적으로 움직이는 대규모 시스템 운영에 유리합니다.
- 추천 상황: 복잡한 멀티 에이전트 워크플로우를 운영하거나, 사내 표준 인프라가 이미 쿠버네티스일 때.


###  DeployAgent 예제 사용법 
1. DeployAgent 폴더에 배포 소스를를 만들고 아래 셀을 차례로 실행시켜 생성된 2개의 파이썬 소스를 이 폴더안에 복사해놓는다
2. 최상위 폴더에 .env 파일을 만들고 아래 내용을 설정해 넣는다(프로젝트에 자신의 프로젝트 ID를 입력하여 사용한다)
   
GOOGLE_GENAI_USE_VERTEXAI=TRUE

GOOGLE_CLOUD_PROJECT=byounghwa-go

GOOGLE_CLOUD_LOCATION=global

4. Launcher 탭에서 Terminal을 실행 시킨 다음 아래 명령을 실행한다
   
adk run DeployAgent

5.아래와 같이 [[user]] 에 입력하여 결과를 확인해본다.  exit를 입력하면 종료된다

Running agent ..., type exit to exit.

[user]]: 현재 빌보드 HOT 100에 있는 노래 중 몇개만 추천해줘

[user]]: 난 재즈를 좋아하니 재즈풍 노래를 추천해줘

[user]]: 지금까지 네가 추천해준 노래를 모두 목록으로 보여줘

[user]]: exit

In [21]:
# !pip install google-cloud-aiplatform[adk,agent_engines]

In [16]:
# 아래 코드는 여기서 직접 실행시키다
import vertexai

client = vertexai.Client(project="byounghwa-go",location="us-central1",)  # 자신의 project ID로 변경하여 입력
memory_bank = client.agent_engines.create()
print(memory_bank)

INFO:vertexai_genai.agentengines:View progress and logs at https://console.cloud.google.com/logs/query?project=byounghwa-go&query=resource.type%3D%22aiplatform.googleapis.com%2FReasoningEngine%22%0Aresource.labels.reasoning_engine_id%3D%222453943377184423936%22.
INFO:vertexai_genai.agentengines:Agent Engine created. To use it in another session:
INFO:vertexai_genai.agentengines:agent_engine=client.agent_engines.get(name='projects/810458263508/locations/us-central1/reasoningEngines/2453943377184423936')


api_client=<vertexai._genai.agent_engines.AgentEngines object at 0x7fd598c2c5b0> api_async_client=<vertexai._genai.agent_engines.AsyncAgentEngines object at 0x7fd599e90d60> api_resource=ReasoningEngine(
  name='projects/810458263508/locations/us-central1/reasoningEngines/2453943377184423936',
  spec=ReasoningEngineSpec(
    effective_identity='service-810458263508@gcp-sa-aiplatform-re.iam.gserviceaccount.com'
  )
)


In [7]:
%%writefile agent.py

import asyncio
import os
from dotenv import load_dotenv


from google.adk.agents import LlmAgent, Agent
from google.adk.tools.agent_tool import AgentTool
from google.adk.runners import Runner
from google.adk.sessions import VertexAiSessionService
from google.adk.memory import VertexAiMemoryBankService
from google.adk.tools import google_search
from google.genai import types

# 환경 변수 로드
load_dotenv()
PROJECT_ID = os.getenv("GOOGLE_CLOUD_PROJECT")
LOCATION = os.getenv("GOOGLE_CLOUD_LOCATION")
AGENT_ENGINE_ID = "2453943377184423936" # 위에서 복사한 ENGINE_ID

# Vertex AI Agent Engine 기반 세션 & 메모리 설정
session_service_instance = VertexAiSessionService(
    project=PROJECT_ID,
    location=LOCATION
)
memory_service_instance = VertexAiMemoryBankService(
    project=PROJECT_ID,
    location=LOCATION,
    agent_engine_id=AGENT_ENGINE_ID
)

# 1. 검색 전문 에이전트
search_agent  = LlmAgent(
    name="SearchAgent",
    model="gemini-2.5-flash",
    instruction="당신은 웹 검색 전문가입니다. 사용자 질문에 대한 최신 정보를 Google 검색 도구를 사용하여 답변하세요.",
    description="Google 검색을 통해 정보를 찾는 검색 에이전트입니다.",
    tools=[google_search],
)

# 2. 오케스트레이션 에이전트 (에이전트를 도구로 사용 Agent-as-a-Tool)
chatbot_agent = LlmAgent(
    name="ConversationAgent",
    model="gemini-2.5-flash",
    instruction="""
    당신은 사용자와 다중 턴 대화를 나누며 정보를 기억하고 활용하는 친근한 챗봇입니다.
    이전 대화 내용을 기억하여 사용자의 질문에 맥락에 맞게 답변하세요.
   
    단순한 질문의 경우 사용자 질문에 직접 응대하고, 최신 정보가 필요한 질문의 경우 'SearchAgent' 도구를 사용하여 검색하세요.
    과거의 대화나 학습된 정보를 활용하여 답변에 참고하세요.
    """,
    description="대화형 챗봇 에이전트입니다.",
    tools=[AgentTool(agent=search_agent)]
)

# ==========================================================
# ★ 중요: ADK CLI(adk run, adk web)가 이 에이전트를 인식하도록 설정
# ==========================================================
root_agent = chatbot_agent 
# ==========================================================


# Runner 설정 및 실행 함수
async def main():
    user_id = "user_123"

    # 세션 생성 (VertexAISessionService는 session_id를 자동으로 생성하고 반환합니다.)
    session = await session_service_instance.create_session(
        app_name=AGENT_ENGINE_ID,
        user_id=user_id
    )

    runner = Runner(
        agent=chatbot_agent,
        app_name=AGENT_ENGINE_ID,
        session_service=session_service_instance,
        memory_service=memory_service_instance,
    )

    print(f"챗봇이 시작되었습니다. (세션 ID: {session.id}, 사용자 ID: {user_id})")
    print("대화를 종료하려면 'exit'을 입력하세요.")

    while True:
        user_query = input("\n사용자👨‍💻 > ")
        if user_query.lower() == 'exit':
            print("챗봇을 종료합니다. 안녕히 가세요!")
            break

        content = types.Content(role='user', parts=[types.Part(text=user_query)])

        final_text_responses = ""
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=content
        ):
            if event.is_final_response() and event.content.parts:
                final_text_responses = event.content.parts[0].text
                print(f"챗봇✨ > {final_text_responses}")            
       
        # 각 턴이 끝날 때, 업데이트된 세션 내용을 MemoryService에 추가하여 기억 생성
        updated_session = await session_service_instance.get_session(
            app_name=AGENT_ENGINE_ID,
            user_id=user_id,
            session_id=session.id
        )
        if updated_session:
            await memory_service_instance.add_session_to_memory(updated_session)

if __name__ == "__main__":
    asyncio.run(main())

Overwriting agent.py


In [2]:
# 배포하기
# 아래 코드는 여기서 직접 실행시키다

import os
from dotenv import load_dotenv
import vertexai
from agent import chatbot_agent
from vertexai import agent_engines

from google.cloud import storage

# 환경 변수 가져오기
PROJECT_ID = os.getenv("GOOGLE_CLOUD_PROJECT")
LOCATION = os.getenv("GOOGLE_CLOUD_LOCATION", "us-central1") # 기본값 설정

# 1. 환경 설정
# PROJECT_ID = "byounghwa-go"
# LOCATION = "asia-northeast3"
BUCKET_NAME = f"{PROJECT_ID}-agent-engine-bucket" # 중복 방지를 위한 버킷 이름

# 2. Cloud Storage 작업 (버킷 생성 및 업로드)
storage_client = storage.Client(project=PROJECT_ID)

# 버킷이 없으면 생성
bucket = storage_client.lookup_bucket(BUCKET_NAME)
if not bucket:
    bucket = storage_client.create_bucket(BUCKET_NAME, location="us-central1")
    print(f"버킷 생성됨: {BUCKET_NAME}")
BUCKET_NAME    

버킷 생성됨: byounghwa-go-agent-engine-bucket


'byounghwa-go-agent-engine-bucket'

In [17]:
LOCATION = "us-central1"

In [18]:
import google.cloud.aiplatform as aiplatform
import google.adk as adk

print(f"1. Vertex AI SDK 버전: {aiplatform.__version__}")
print(f"2. Google ADK 버전: {adk.__version__}")


1. Vertex AI SDK 버전: 1.147.0
2. Google ADK 버전: 1.29.0


In [24]:
STAGING_BUCKET_NAME = f"gs://{BUCKET_NAME}"

# 1단계: Agent Engine 초기화
vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=STAGING_BUCKET_NAME
)

print(f"Vertex AI 초기화 완료. 스테이징 버킷: {STAGING_BUCKET_NAME}")

# 2단계: 에이전트를 AdkApp 객체로 래핑
app = agent_engines.AdkApp(
    agent=chatbot_agent,
    enable_tracing=True,
)

# 3단계: Agent Engine에 배포
# Agent Engine이 환경을 구축할 때 필요한 라이브러리를 지정합니다.
remote_app = agent_engines.create(
    agent_engine=app,
    # requirements=["google-cloud-aiplatform[agent_engines]", "google-adk", "cloudpickle==3.1.2", "python-dotenv",'pydantic==2.12.5'],
    requirements=[
        "google-cloud-aiplatform>=1.147.0", # vertexai 모듈을 포함하는 패키지
        "google-adk", 
        "cloudpickle==3.1.2", 
        "python-dotenv"
    ],
    gcs_dir_name='Chatbot/agent_engine',
    display_name='adk_multiturn_agent',
    description='ADK기반 Multi-turn 챗봇 에이전트입니다.',
    extra_packages=["agent.py"],
    
)

# 배포된 에이전트의 고유 식별자: projects/.../reasoningEngines/{RESOURCE_ID}
print(f"리소스 이름 (Resource Name): {remote_app.resource_name}")
# 장시간 소요

In [ ]:
# 쿼리하기
# 아래 코드는 여기서 직접 실행시키다
# projects/810458263508/locations/us-central1/reasoningEngines/5920307715376152576/operations/7199284697644400640

import asyncio
from vertexai import agent_engines

async def main():
    app = agent_engines.get("projects/810458263508/locations/us-central1/reasoningEngines/5920307715376152576/operations/7199284697644400640")
    user_id = "remote_user_123"

    session = await app.async_create_session(user_id=user_id)
    session_id = session["id"]
    print(f"챗봇이 시작되었습니다. (세션 ID: {session_id}, 사용자 ID: {user_id})")
    print("대화를 종료하려면 'exit'를 입력하세요.")

    # 3단계: 사용자와의 실시간 대화 루프
    while True:
        user_query = await asyncio.to_thread(input, "사용자👨‍💻 > ")
        if user_query.lower() == 'exit':
            print("챗봇을 종료합니다.")
            break

        # 종료 명령어 확인
        if user_query.lower() == "exit":
            print("챗봇을 종료합니다. 안녕히 가세요!")
            break

        # 4단계: 쿼리 전송 및 응답 스트리밍
        events = []

        # 동일한 세션 ID를 계속 재사용하여 대화의 맥락을 유지
        async for event in app.async_stream_query(
            user_id=user_id,
            session_id=session_id,
            message=user_query
        ):
            events.append(event)
           
        final_text_responses = [
            e for e in events
            if e.get("content", {}).get("parts", [{}])[0].get("text")
            and not e.get("content", {}).get("parts", [{}])[0].get("function_call")
        ]
        if final_text_responses:
            response = final_text_responses[0]["content"]["parts"][0]["text"]
            print(f"챗봇✨ > {response}")

# 스크립트 실행
if __name__ == "__main__":
    asyncio.run(main())

In [ ]:
# The end